# Session 7: pandas I, load, look and filter

**KLAi | AI and Data Science Professional**

In this lab you load the Lakeside Mart orders file (fictional, generated data), look at it before you trust it, filter it, and save a data profile. Each step has the same rhythm: **PREDICT**, run, **READ**, then **CHANGE one thing**. A cell that says `None  # your code here` is yours to fill. The check cells say OK or NOT YET, and they keep running while an answer is still `None`. Keep your answers to each prediction in the grey boxes.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
# Helper for the checks. Run this cell once. You do not need to read it.
# attempt() runs a check on your answer. If the answer is not written yet it says NOT YET instead of stopping with an error.
def attempt(label, expr, expected, tol=0.01):
    try:
        value = eval(expr)
        want = eval(expected) if isinstance(expected, str) else expected
    except Exception:
        value, want = None, 0
    check(label, value, want, tol)

---
## Step 1: Load the file and look at its shape

read_csv reads a CSV file into a DataFrame. shape gives the number of rows, then the number of columns.

**PREDICT first.** The file holds one record for each till order. Roughly how many rows do you expect: hundreds, thousands or hundreds of thousands?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: pd.read_csv('orders.csv')
import pandas as pd

orders = None  # your code here: read orders.csv into a DataFrame

In [ ]:
try:
    print(orders.shape)
    print(orders.dtypes)
    print(orders.head(3))
except Exception:
    print("Not yet: finish the cell above, then run this cell again.")
attempt('number of rows', 'orders.shape[0]', 30229, 0)
attempt('number of columns', 'orders.shape[1]', 7, 0)

**READ the output.** Which column is stored as text but holds dates? What data type does pandas call text? (It is object, and pandas 3 and later shows it as str.)

> My answer: 

**CHANGE one thing.** Change head(3) to head(8) and run the cell. What do you see about the index?

> What I saw: 

---
## Step 2: Missing values and a summary

isna() marks each empty cell True and sum() counts the Trues for each column. describe() summarises a numeric variable.

**PREDICT first.** Which column do you expect to have missing values? (Remember the walk-in shoppers from Session 2.) Will the mean order value be bigger or smaller than the median?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: chain isna() and sum(), then pick the customer_id entry. Use .mean() and .median() on the column order_value_ugx
n_missing = None  # your code here: the number of missing customer_id values
mean_value = None  # your code here
median_value = None  # your code here

In [ ]:
try:
    print(orders.isna().sum())
    print(orders["order_value_ugx"].describe())
except Exception:
    print("Not yet: finish the cell above, then run this cell again.")
attempt('missing customer_id', 'n_missing', 9171, 0)
attempt('mean order value', 'mean_value', 47491.29, 0.5)
attempt('median order value', 'median_value', 23200.0, 0.5)

**READ the output.** The mean is about twice the median. What does that tell you about the largest orders?

> My answer: 

**CHANGE one thing.** Add .info() on its own line: orders.info(). Find the line that shows how many customer_id values are filled.

> What I saw: 

---
## Step 3: Select columns and filter with a Boolean mask

A comparison such as orders["branch_id"] == "B04" builds a mask of True and False, one per row. Putting the mask inside square brackets after the DataFrame keeps the True rows.

**PREDICT first.** Branch B04 is Jinja. About how many of the orders do you expect to come from one branch, out of eight branches?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: mask = a comparison with ==; then orders[mask]. Take a list of two names for two_columns
two_columns = None  # your code here: a DataFrame with only order_id and order_value_ugx
mask = None  # your code here: the comparison for branch B04
jinja = None  # your code here: orders where the mask is True

In [ ]:
try:
    print(two_columns.shape)
    print(mask.sum(), len(jinja))
except Exception:
    print("Not yet: finish the cell above, then run this cell again.")
attempt('two columns kept', 'two_columns.shape[1]', 2, 0)
attempt('Jinja orders', 'len(jinja)', 3562, 0)
attempt('mask sum equals row count', 'mask.sum()', 'len(jinja)', 0)

**READ the output.** Why does mask.sum() give the same number as len(jinja)?

> My answer: 

**CHANGE one thing.** Filter for branch B01 instead. How many orders does it have? Change it back to B04 afterwards.

> What I saw: 

---
## Step 4: Two conditions, and query

Join two masks with & (and) or | (or). Each condition goes in its own round brackets. query() writes the condition as text.

**PREDICT first.** Jinja orders paid by Mobile money: will the count be bigger or smaller than the number of Jinja orders? And will "Jinja or Mobile money" be bigger or smaller than "Jinja and Mobile money"?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: (test one) & (test two); then (test one) | (test two). query("branch_id == 'B04' and payment_method == 'Mobile money'")
n_both = None  # your code here: Jinja AND Mobile money
n_either = None  # your code here: Jinja OR Mobile money
n_query = None  # your code here: the same as n_both, written with query

In [ ]:
print(n_both, n_either, n_query)
attempt('Jinja and Mobile money', 'n_both', 1695, 0)
attempt('Jinja or Mobile money', 'n_either', 16179, 0)
attempt('query gives the same as &', 'n_query', 'n_both', 0)

**READ the output.** Which is larger, n_both or n_either? Why must that always be true?

> My answer: 

**CHANGE one thing.** On purpose, write the first filter with the word and instead of &, and run it. Read the last line of the error. Then fix it.

> What I saw: 

**CHANGE one thing.** Try three more filters and read each count: `orders["branch_id"].isin(["B01", "B04"])`, `orders["order_value_ugx"].between(10000, 50000)` and `orders["payment_method"].str.contains("Mobile")`. Put each inside `orders[...]` and use `len`. Then run `orders["branch_id"].nunique()`.

> What I saw: 

---
## Step 5: Sort and count

sort_values orders the rows. head(10) keeps ten. value_counts() counts each different value of a Series.

**PREDICT first.** Will the largest order be near the mean order value, or many times larger? Which payment method do you expect to be the most common?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: sort_values('order_value_ugx', ascending=False).head(10); value_counts() on payment_method
top10 = None  # your code here: the ten largest orders
counts = None  # your code here: the number of orders for each payment method

In [ ]:
try:
    print(top10[["order_id", "branch_id", "order_value_ugx"]])
    print(counts)
except Exception:
    print("Not yet: finish the cell above, then run this cell again.")
attempt('largest order value', 'top10["order_value_ugx"].iloc[0]', 3104000, 0)
attempt('sum of the ten largest', 'top10["order_value_ugx"].sum()', 26796945, 0)
attempt('Mobile money orders', 'counts["Mobile money"]', 14312, 0)
attempt('counts add up to all orders', 'counts.sum()', 30229, 0)

**READ the output.** How many times larger than the median order is the largest order? Is it a sale or a typing error? What would you ask the data owner?

> My answer: 

**CHANGE one thing.** Use ascending=True. What is the smallest order value?

> What I saw: 

---
## Step 6: Pick rows with loc and iloc

iloc picks by position (counting from 0, end left out). loc picks by label (the index value, end included). After a filter the labels are no longer 0, 1, 2.

**PREDICT first.** jinja.iloc[0] is the first row of jinja. What do you think its index label is: 0, or something else?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: jinja.index[0] is the first label. Use iloc with a position and loc with a label
first_label = None  # your code here: the index label of the first row of jinja
value_by_position = None  # your code here: order_value_ugx of the row at position 0
value_by_label = None  # your code here: order_value_ugx of the row with that label

In [ ]:
print(first_label, value_by_position, value_by_label)
attempt('first label of jinja', 'first_label', 14060, 0)
attempt('value of the first Jinja row', 'value_by_position', 185865, 0)
attempt('loc gives the same row', 'value_by_label', 'value_by_position', 0)

**READ the output.** Why would jinja.loc[0] give an error?

> My answer: 

**CHANGE one thing.** Run jinja.loc[0] in a new cell and read the last line. Then delete the cell.

> What I saw: 

---
## Step 7: Build and save a data profile

A data profile has one row for each variable: its data type, how many values are missing, and the share missing. You keep it, and you can build the same profile for any file.

**PREDICT first.** How many rows will the profile have, and which variable will have a share missing above zero?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: a DataFrame from a dictionary; dtypes.astype(str) gives readable names; share = missing / number of rows * 100
profile = None  # your code here: a DataFrame with columns dtype, missing and missing_pct, one row per variable

In [ ]:
try:
    profile.to_csv("orders_profile.csv")
    print(profile)
except Exception:
    print("Not yet: finish the cell above, then run this cell again.")
attempt('profile rows', 'len(profile)', 7, 0)
attempt('missing share of customer_id', 'profile.loc["customer_id", "missing_pct"]', 30.3, 0.05)
attempt('variables with missing values', '(profile["missing"] > 0).sum()', 1, 0)

**READ the output.** If a colleague sent you this profile of a file you had never seen, what would you know about it?

> My answer: 

**CHANGE one thing.** Add one more column, n_unique, with the number of different values in each variable (orders.nunique()). Which variable has the most?

> What I saw: 

## Check against Sheets

Open `orders.csv` in Google Sheets. Use a filter to reproduce the Jinja count from step 3 and the Jinja and Mobile money count from step 4. Write one sentence about anything that did not match.

> My sentence: 

---
## Finish

Run the cell below. It should say **All 21 checks pass**. Then download `orders_profile.csv` from the file panel on the left, and save the notebook to your Drive.

In [ ]:
show_checks()